In [13]:
import pandas as pd
from tqdm import tqdm
import pathlib
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import train_test_split
pd.set_option('display.max_columns', None)

In [14]:
df = pd.read_csv('../data/trusted/base_model_gols.csv')

### Função para avaliar o modelo

In [15]:
def avaliar_modelo(nome_modelo, y_teste, previsao):
    r2 = r2_score(y_teste, previsao)
    RSME = np.sqrt(mean_squared_error(y_teste, previsao))
    return f'Modelo {nome_modelo}:\nR²:{r2:.2%}\nRSME:{RSME:.2f}'

In [16]:
df.head(1)

,id_player_casa,gols_casa,id_time_casa,id_player_fora,gols_fora,id_time_fora,gols_total
0,603262,3,21,603265,3,6,6


#### Modelos
##### Escolha dos Modelos a Serem Testados para número de gols
- RandomForest
- LinearRegression
- Extra Tree

In [17]:
modelo_rf = RandomForestRegressor()
modelo_lr = LinearRegression()
modelo_et = ExtraTreesRegressor()

modelos = {'RandomForest': modelo_rf,
          'LinearRegression': modelo_lr,
          'ExtraTrees': modelo_et,
          }

y = df['gols_total'] # Alvo do modelo
X = df.drop(['gols_total', 'gols_casa', 'gols_fora'], axis=1) # Features sem dados pós jogo


#### Transformar IDs em features categóricas

In [18]:
# Copia as features
X_encoded = X.copy()

features_categoricas = ['id_player_casa', 'id_time_casa', 'id_player_fora', 'id_time_fora']

for col in features_categoricas:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col])


#### Separação dos dados de treino e teste

In [19]:
X_treino, X_teste, y_train, y_test = train_test_split(X_encoded, y, random_state=42, test_size=0.2)

for nome_modelo, modelo in tqdm(modelos.items()):
    # Treinar
    print('Iniciando treino')
    print(f'Modelo: {nome_modelo}')
    modelo.fit(X_treino, y_train)

    # Testar
    print('Iniciando previsão')
    print(f'Modelo: {nome_modelo}')
    previsao = modelo.predict(X_teste)
    print(avaliar_modelo(nome_modelo, y_test, previsao))

  0%|          | 0/3 [00:00<?, ?it/s]

Iniciando treino
Modelo: RandomForest
Iniciando previsão
Modelo: RandomForest


 33%|███▎      | 1/3 [00:56<01:52, 56.16s/it]

Modelo RandomForest:
R²:63.91%
RSME:1.44
Iniciando treino
Modelo: LinearRegression
Iniciando previsão
Modelo: LinearRegression
Modelo LinearRegression:
R²:1.17%
RSME:2.38
Iniciando treino
Modelo: ExtraTrees
Iniciando previsão
Modelo: ExtraTrees


100%|██████████| 3/3 [01:16<00:00, 25.36s/it]

Modelo ExtraTrees:
R²:71.55%
RSME:1.28
